# 01 — Diagnosis Kualitas Data
### Milestone M2 · Exploratory Data Analysis (KMTI21133)

| | |
|---|---|
| **Nama tim** | Kelompok Bakery Sales |
| **Anggota** | Alfandi Panca Saputra (0110225026) <br>Muhammad Sultan Ridho (0110225099)<br>Dzakwan Abdur Rasyid (0110225036)<br>Mumtaaz Abdurrahman (0110225164) |
| **Topik proyek** | Eksplorasi Data Penjualan Bakery |
| **Tanggal** | 9 November 2026 |

**Notebook ini mengisi:** dimensi **Kelengkapan diagnosis kualitas data** (bobot 25) pada Rubrik M2.

Tujuan notebook ini **hanya mendiagnosis** — belum memperbaiki apa pun. Pembersihan
dikerjakan di `02_cleaning.ipynb`. Memisahkan keduanya penting supaya temuan dapat
diperiksa ulang tanpa tercampur dengan perubahan yang sudah terlanjur dilakukan.

Periksa **kelima** dimensi di bawah. Memeriksa hanya nilai kosong dan duplikat membuat
dimensi ini bernilai 2 ke bawah.

---
> **Sebelum mengumpulkan:** jalankan **Kernel → Restart & Run All**. Notebook yang tidak dapat
> berjalan dari awal sampai akhir dianggap tidak memenuhi dimensi *ketepatan implementasi*.

In [4]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

DATA  = '#0E7490'   # warna utama
SOROT = '#C2410C'   # penyorot
REDUP = '#5B6672'

# Path relatif terhadap folder notebooks/ -- JANGAN diganti menjadi path absolut.
AKAR      = Path('..')
RAW       = AKAR / 'data' / 'raw'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES   = AKAR / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## 0. Muat data mentah

Baca dari `data/raw`. Berkas ini **tidak pernah diubah** sepanjang proyek.

In [6]:
BERKAS = RAW / '/content/drive/MyDrive/EDA/Bakery sales (2) (1).csv'     # TODO: ganti nama berkas

assert BERKAS.exists(), (
    f'Berkas tidak ditemukan: {BERKAS}\n'
    'Periksa apakah data sudah diletakkan di data/raw dan namanya sudah benar.'
)
df = pd.read_csv(BERKAS)
print(f'{df.shape[0]} baris x {df.shape[1]} kolom')
df.head()

234005 baris x 7 kolom


,Unnamed: 0,date,time,ticket_number,article,Quantity,unit_price
0,0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,"0,90 €"
1,1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,"1,20 €"
2,4,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,"1,20 €"
3,5,2021-01-02,09:14,150041.0,PAIN,1.0,"1,15 €"
4,8,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,"1,20 €"


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234005 entries, 0 to 234004
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     234005 non-null  int64  
 1   date           234005 non-null  object 
 2   time           234005 non-null  object 
 3   ticket_number  234005 non-null  float64
 4   article        234005 non-null  object 
 5   Quantity       234005 non-null  float64
 6   unit_price     234005 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 12.5+ MB


## 1. Kelengkapan — nilai yang hilang

Jangan berhenti pada jumlahnya. Yang dinilai adalah apakah Anda memeriksa **polanya**:
apakah kekosongan berkaitan dengan kolom lain?

In [8]:
kosong = pd.DataFrame({
    'jumlah' : df.isna().sum(),
    'persen' : (df.isna().mean() * 100).round(2),
})
kosong = kosong[kosong.jumlah > 0].sort_values('jumlah', ascending=False)
print(kosong.to_string() if len(kosong) else 'Tidak ada nilai kosong.')

Tidak ada nilai kosong.


In [16]:
# Karena hasil diagnosis pada langkah sebelumnya menunjukkan bahwa tidak ada nilai kosong (missing values)
# pada dataset ini, maka analisis pola kekosongan pada sel ini dilewati (di-bypass).

print("Langkah dilewati: Tidak ada kolom yang memiliki nilai kosong untuk dianalisis polanya.")

Langkah dilewati: Tidak ada kolom yang memiliki nilai kosong untuk dianalisis polanya.


```markdown
**Temuan:**

Berdasarkan hasil diagnosis kelengkapan data di sel `sel007`, **tidak ditemukan adanya nilai kosong (missing values) sama sekali pada dataset ini** (semua kolom memiliki kelengkapan 100%). Oleh karena itu, analisis pola kekosongan pada langkah ini tidak diperlukan, dan tidak ada tindakan imputasi atau penanganan nilai kosong yang perlu direncanakan pada tahap pembersihan data (`02_cleaning.ipynb`).
```

## 2. Keunikan — baris kembar

In [18]:
print('Duplikat seluruh kolom :', df.duplicated().sum())

# Menggunakan 'ticket_number' sebagai kunci transaksi
KUNCI = ['ticket_number']
print('Duplikat pada kolom kunci (ticket_number):', df.duplicated(subset=KUNCI).sum())
print('\nCatatan: Duplikat pada ticket_number wajar karena satu transaksi (satu nomor tiket) dapat berisi beberapa item (article) berbeda.')

Duplikat seluruh kolom : 0
Duplikat pada kolom kunci (ticket_number): 97554

Catatan: Duplikat pada ticket_number wajar karena satu transaksi (satu nomor tiket) dapat berisi beberapa item (article) berbeda.


## 3. Konsistensi — penulisan kategori

Beda kapitalisasi dan spasi di ujung membuat satu kategori terpecah menjadi beberapa.

In [19]:
kategorikal = df.select_dtypes(include=['object', 'string']).columns.tolist()

for k in kategorikal:
    n_mentah  = df[k].nunique()
    n_rapi    = df[k].astype(str).str.strip().str.lower().nunique()
    tanda = '  <-- PERIKSA' if n_rapi < n_mentah else ''
    print(f'{k:24s} unik: {n_mentah:4d}   setelah dirapikan: {n_rapi:4d}{tanda}')

date                     unik:  600   setelah dirapikan:  600
time                     unik:  683   setelah dirapikan:  683
article                  unik:  149   setelah dirapikan:  149
unit_price               unik:  123   setelah dirapikan:  123


In [22]:
KOLOM = 'article'          # Menggunakan kolom 'article' yang bertipe kategorikal
print(sorted(df[KOLOM].dropna().unique()))

['.', '12 MACARON', 'ARMORICAIN', 'ARTICLE 295', 'BAGUETTE', 'BAGUETTE APERO', 'BAGUETTE GRAINE', 'BANETTE', 'BANETTINE', 'BOISSON 33CL', 'BOTTEREAU', 'BOULE 200G', 'BOULE 400G', 'BOULE POLKA', 'BRIOCHE', 'BRIOCHE DE NOEL', 'BRIOCHETTE', 'BROWNIES', 'BUCHE 4PERS', 'BUCHE 6PERS', 'BUCHE 8PERS', 'CAFE OU EAU', 'CAKE', 'CAMPAGNE', 'CARAMEL NOIX', 'CEREAL BAGUETTE', 'CHAUSSON AUX POMMES', 'CHOCOLAT', 'CHOU CHANTILLY', 'COMPLET', 'COOKIE', 'COUPE', 'CROISSANT', 'CROISSANT AMANDES', 'CRUMBLE', 'CRUMBLECARAMEL OU PISTAE', 'DELICETROPICAL', 'DEMI BAGUETTE', 'DEMI PAIN', 'DIVERS BOISSONS', 'DIVERS BOULANGERIE', 'DIVERS CONFISERIE', 'DIVERS PATISSERIE', 'DIVERS SANDWICHS', 'DIVERS VIENNOISERIE', 'DOUCEUR D HIVER', 'ECLAIR', 'ECLAIR FRAISE PISTACHE', 'ENTREMETS', 'FICELLE', 'FINANCIER', 'FINANCIER X5', 'FLAN', 'FLAN ABRICOT', 'FONDANT CHOCOLAT', 'FORMULE PATE', 'FORMULE PLAT PREPARE', 'FORMULE SANDWICH', 'FRAISIER', 'FRAMBOISIER', 'GACHE', 'GAL FRANGIPANE 4P', 'GAL FRANGIPANE 6P', 'GAL POIRE CHOC

## 4. Validitas — nilai yang mustahil dan tipe data

Dua hal: apakah tipe datanya benar, dan apakah nilainya mungkin terjadi di dunia nyata.

In [23]:
print('TIPE DATA YANG PERLU DIPERIKSA')
for k in df.columns:
    if df[k].dtype == object:
        contoh = df[k].dropna().iloc[0] if df[k].notna().any() else ''
        print(f'  {k:24s} object   contoh: {contoh!r}')

TIPE DATA YANG PERLU DIPERIKSA
  date                     object   contoh: '2021-01-02'
  time                     object   contoh: '08:38'
  article                  object   contoh: 'BAGUETTE'
  unit_price               object   contoh: '0,90 €'


In [24]:
# Rentang nilai numerik -- cari nilai yang tidak mungkin (negatif, nol, terlalu besar)
num = df.select_dtypes('number')
print(num.describe().T[['min','25%','50%','75%','max']].round(2).to_string())

                    min       25%       50%       75%       max
Unnamed: 0          0.0  127979.0  254573.0  382911.0  511395.0
ticket_number  150040.0  184754.0  218807.0  253927.0  288913.0
Quantity         -200.0       1.0       1.0       2.0     200.0


In [26]:
# Memeriksa nilai yang tidak masuk akal pada kolom Quantity
print('Quantity negatif                :', (df['Quantity'] < 0).sum())
print('Quantity bernilai nol          :', (df['Quantity'] == 0).sum())
print('Quantity terlalu besar (> 50)  :', (df['Quantity'] > 50).sum())

Quantity negatif                : 1295
Quantity bernilai nol          : 0
Quantity terlalu besar (> 50)  : 2


> **Kolom tanggal:** bila format penulisannya hari/bulan/tahun, `pd.to_datetime` secara
> bawaan akan membacanya sebagai bulan/hari/tahun dan **tidak memunculkan pesan kesalahan**.
> Setiap tanggal yang harinya ≤ 12 akan tertukar secara diam-diam. Gunakan
> `format='%d/%m/%Y'`, lalu periksa `.min()`, `.max()`, dan jumlah bulan yang muncul.

## 5. Akurasi — konsistensi antarkolom

Dimensi yang paling sering dilewatkan. Cari hubungan yang **harus** berlaku di data Anda,
lalu uji apakah benar-benar berlaku.

In [27]:
# Pemeriksaan Konsistensi: Apakah satu jenis produk (article) memiliki beberapa harga satuan (unit_price) yang berbeda?
harga_per_produk = df.groupby('article')['unit_price'].nunique()
produk_beda_harga = harga_per_produk[harga_per_produk > 1]

print('Jumlah produk yang memiliki lebih dari satu variasi harga satuan:', len(produk_beda_harga))
if len(produk_beda_harga) > 0:
    print('\nBeberapa contoh produk dengan variasi harga tersebut (Top 5):')
    print(produk_beda_harga.head())

Jumlah produk yang memiliki lebih dari satu variasi harga satuan: 78

Beberapa contoh produk dengan variasi harga tersebut (Top 5):
article
BAGUETTE           3
BAGUETTE GRAINE    3
BANETTE            3
BANETTINE          3
BOULE 200G         3
Name: unit_price, dtype: int64


## 6. Ringkasan temuan

Isi tabel ini. Tabel inilah yang disalin ke Data Quality Report dan menjadi dasar
urutan langkah di `02_cleaning.ipynb`.

| # | Masalah | Dimensi | Jumlah / bukti | Rencana penanganan |
|---|---------|---------|----------------|--------------------|
| 1 |Duplikat pada kolom kunci (ticket_number) |Keunikan | 97.554 duplikat (Duplikat seluruh kolom: 0) |Mempertahankan duplikat karena wajar satu transaksi berisi beberapa item/artikel berbeda. |
| 2 |Format nama item/kategori tidak bersih (karakter tidak standar/titik) |Konsistensi |Muncul nilai seperti . dan ARTICLE 295 pada list nilai unik kolom |Melakukan pembersihan string (stripping/lowercasing) dan menghapus/memperbaiki item yang berupa simbol/placeholder.|
| 3 |Tipe data tidak sesuai (tipe object pada tanggal, waktu, harga) |Validitas |Kolom date, time, article, dan unit_price bertipe object (contoh unit_price: '0,90 €') | Mengonversi date & time ke tipe datetime serta membersihkan format mata uang pada unit_price menjadi float.|
| 4 | Nilai Quantity tidak masuk akal (negatif & terlalu besar)| Validitas|Quantity negatif: 1.295, Quantity > 50: 2 (rentang -200 hingga 200) | Menghapus atau memisahkan baris dengan nilai Quantity negatif (transaksi retur/pembatalan) serta memeriksa pencatatan ekstrim (>50).|
| 5 | Variasi harga satuan pada jenis produk yang sama|Akurasi | 78 produk memiliki lebih dari 1 variasi harga satuan (misal: BAGUETTE, BANETTE masing-masing punya 3 variasi harga) | Menginvestigasi apakah variasi harga disebabkan oleh perubahan harga seiring waktu/diskon atau kesalahan input data. |

## Checklist sebelum mengumpulkan

- [x] Kelima dimensi diperiksa, bukan hanya nilai kosong dan
duplikat.
- [x] Pola kekosongan diperiksa, bukan sekadar jumlahnya.
- [x] Minimal satu pemeriksaan konsistensi antarkolom ditulis.
- [x] Tabel ringkasan temuan terisi lengkap.
- [ ] `data/raw` tidak berubah — notebook ini hanya membaca.
- [x] Restart & Run All berjalan tanpa galat.